In [1]:
import pandas as pd
import numpy as np
import sklearn
from google.colab import files

SEED = 42
np.random.seed(SEED)

print("Setup ready")
print("scikit-learn version:", sklearn.__version__)

# Upload the original dataset and supplied test labels.
uploaded = files.upload()

Setup ready
scikit-learn version: 1.6.1


Saving online_shoppers_intention.csv to online_shoppers_intention.csv
Saving y_test (1).csv to y_test (1).csv


In [2]:
import io
from sklearn.model_selection import train_test_split

df = pd.read_csv(io.BytesIO(uploaded["online_shoppers_intention.csv"]))
supplied_y_test = pd.read_csv(
    io.BytesIO(uploaded["y_test (1).csv"])
)["Revenue"].to_numpy()

print("Original shape:", df.shape)
print("Duplicate rows:", df.duplicated().sum())

# Remove exact duplicates before splitting.
df = df.drop_duplicates().reset_index(drop=True)

X = df.drop(columns="Revenue")
y = df["Revenue"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=SEED
)

assert np.array_equal(y_test.to_numpy(), supplied_y_test), (
    "Test labels do not match. Stop here and check the input files."
)

print("\nTraining shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("\nTraining class counts:")
print(y_train.value_counts().sort_index())
print("\nTest labels match the supplied file.")

Original shape: (12330, 18)
Duplicate rows: 125

Training shape: (9764, 17)
Test shape: (2441, 17)

Training class counts:
Revenue
0    8238
1    1526
Name: count, dtype: int64

Test labels match the supplied file.


In [3]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

# These numeric codes represent categories, not quantities.
categorical_cols = [
    "Month", "OperatingSystems", "Browser",
    "Region", "TrafficType", "VisitorType", "Weekend"
]

numerical_cols = [
    col for col in X_train.columns
    if col not in categorical_cols
]

numerical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    ))
])

preprocessor = ColumnTransformer([
    ("numerical", numerical_pipeline, numerical_cols),
    ("categorical", categorical_pipeline, categorical_cols)
])

print("Numerical features:", len(numerical_cols))
print("Categorical features:", len(categorical_cols))
print("Preprocessing defined — not fitted yet.")

Numerical features: 10
Categorical features: 7
Preprocessing defined — not fitted yet.


In [4]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=SEED
)

baseline = Pipeline([
    ("preprocessing", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=100,
        random_state=SEED,
        n_jobs=-1
    ))
])

scoring = {
    "f1": "f1",
    "precision": "precision",
    "recall": "recall",
    "roc_auc": "roc_auc",
    "pr_auc": "average_precision"
}

baseline_scores = cross_validate(
    baseline,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    return_train_score=True,
    n_jobs=1,
    error_score="raise"
)

baseline_summary = pd.DataFrame([
    {
        "Metric": metric,
        "Train mean": baseline_scores[f"train_{metric}"].mean(),
        "CV mean": baseline_scores[f"test_{metric}"].mean(),
        "CV std": baseline_scores[f"test_{metric}"].std(ddof=1)
    }
    for metric in scoring
])

print(baseline_summary.round(4).to_string(index=False))

   Metric  Train mean  CV mean  CV std
       f1         1.0   0.6137  0.0129
precision         1.0   0.7727  0.0184
   recall         1.0   0.5092  0.0164
  roc_auc         1.0   0.9227  0.0037
   pr_auc         1.0   0.7369  0.0198


In [5]:
from sklearn.base import clone

# Preserve the baseline as the first comparison row.
results = [{
    "Variety": "Baseline",
    "Parameters": str(baseline.named_steps["model"].get_params()),
    "Train F1": baseline_scores["train_f1"].mean(),
    **{
        f"CV {metric}": baseline_scores[f"test_{metric}"].mean()
        for metric in scoring
    },
    "CV F1 std": baseline_scores["test_f1"].std(ddof=1)
}]

varieties = {
    "Balanced weights": {
        "class_weight": "balanced"
    },
    "Balanced subsample": {
        "class_weight": "balanced_subsample"
    },
    "Regularized trees": {
        "max_depth": 12,
        "min_samples_leaf": 5
    },
    "Regularized + balanced": {
        "max_depth": 12,
        "min_samples_leaf": 5,
        "class_weight": "balanced"
    }
}

# Keep definitions for later selection and final fitting.
candidate_models = {"Baseline": clone(baseline)}

for name, params in varieties.items():
    print(f"Evaluating: {name}", flush=True)

    candidate = Pipeline([
        ("preprocessing", clone(preprocessor)),
        ("model", RandomForestClassifier(
            n_estimators=100,
            random_state=SEED,
            n_jobs=-1,
            **params
        ))
    ])

    candidate_models[name] = candidate

    scores = cross_validate(
        candidate,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        return_train_score=True,
        n_jobs=1,
        error_score="raise"
    )

    results.append({
        "Variety": name,
        "Parameters": str(candidate.named_steps["model"].get_params()),
        "Train F1": scores["train_f1"].mean(),
        **{
            f"CV {metric}": scores[f"test_{metric}"].mean()
            for metric in scoring
        },
        "CV F1 std": scores["test_f1"].std(ddof=1)
    })

comparison = pd.DataFrame(results).sort_values(
    "CV f1", ascending=False
)

display(comparison.drop(columns="Parameters").round(4))

Evaluating: Balanced weights
Evaluating: Balanced subsample
Evaluating: Regularized trees
Evaluating: Regularized + balanced


,Variety,Train F1,CV f1,CV precision,CV recall,CV roc_auc,CV pr_auc,CV F1 std
4,Regularized + balanced,0.7335,0.6668,0.5720,0.8001,0.9235,0.7187,0.0100
0,Baseline,1.0000,0.6137,0.7727,0.5092,0.9227,0.7369,0.0129
1,Balanced weights,1.0000,0.6066,0.7656,0.5026,0.9230,0.7311,0.0180
2,Balanced subsample,1.0000,0.6028,0.7701,0.4954,0.9243,0.7308,0.0194
3,Regularized trees,0.7236,0.5945,0.8286,0.4640,0.9254,0.7494,0.0191


In [7]:
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.under_sampling import RandomUnderSampler

# Same tree settings as the strongest variety so far.
rf_settings = {
    "n_estimators": 100,
    "max_depth": 12,
    "min_samples_leaf": 5,
    "random_state": SEED,
    "n_jobs": -1
}

# Sampling happens only within each CV fold's training portion.
undersampled = ImbPipeline([
    ("preprocessing", clone(preprocessor)),
    ("sampling", RandomUnderSampler(random_state=SEED)),
    ("model", RandomForestClassifier(**rf_settings))
])

# Define a separate preprocessor that excludes PageValues.
preprocessor_no_pagevalues = ColumnTransformer([
    (
        "numerical",
        clone(numerical_pipeline),
        [col for col in numerical_cols if col != "PageValues"]
    ),
    (
        "categorical",
        clone(categorical_pipeline),
        categorical_cols
    )
])

without_pagevalues = Pipeline([
    ("preprocessing", preprocessor_no_pagevalues),
    ("model", RandomForestClassifier(
        **rf_settings,
        class_weight="balanced"
    ))
])

new_varieties = {
    "Regularized + undersampling": undersampled,
    "Regularized + balanced, no PageValues": without_pagevalues
}

for name, candidate in new_varieties.items():
    print(f"Evaluating: {name}", flush=True)
    candidate_models[name] = candidate

    scores = cross_validate(
        candidate,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        return_train_score=True,
        n_jobs=1,
        error_score="raise"
    )

    results.append({
        "Variety": name,
        "Parameters": str(candidate.get_params()),
        "Train F1": scores["train_f1"].mean(),
        **{
            f"CV {metric}": scores[f"test_{metric}"].mean()
            for metric in scoring
        },
        "CV F1 std": scores["test_f1"].std(ddof=1)
    })

comparison = pd.DataFrame(results).sort_values(
    "CV f1", ascending=False
)

display(comparison.drop(columns="Parameters").round(4))

Evaluating: Regularized + undersampling
Evaluating: Regularized + balanced, no PageValues


,Variety,Train F1,CV f1,CV precision,CV recall,CV roc_auc,CV pr_auc,CV F1 std
4,Regularized + balanced,0.7335,0.6668,0.5720,0.8001,0.9235,0.7187,0.0100
7,Regularized + undersampling,0.6730,0.6435,0.5183,0.8499,0.9207,0.7015,0.0109
5,Regularized + undersampling,0.6730,0.6435,0.5183,0.8499,0.9207,0.7015,0.0109
0,Baseline,1.0000,0.6137,0.7727,0.5092,0.9227,0.7369,0.0129
1,Balanced weights,1.0000,0.6066,0.7656,0.5026,0.9230,0.7311,0.0180
2,Balanced subsample,1.0000,0.6028,0.7701,0.4954,0.9243,0.7308,0.0194
3,Regularized trees,0.7236,0.5945,0.8286,0.4640,0.9254,0.7494,0.0191
6,"Regularized + balanced, no PageValues",0.5792,0.4239,0.3204,0.6265,0.7737,0.3823,0.0227
8,"Regularized + balanced, no PageValues",0.5792,0.4239,0.3204,0.6265,0.7737,0.3823,0.0227


In [8]:
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from imblearn.over_sampling import SMOTENC

# Output order: 10 numerical columns, then 7 categorical columns.
before_sampling = ColumnTransformer([
    ("numerical", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), numerical_cols),

    ("categorical", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OrdinalEncoder(
            handle_unknown="use_encoded_value",
            unknown_value=-1
        ))
    ]), categorical_cols)
])

numerical_indices = list(range(len(numerical_cols)))
categorical_indices = list(range(
    len(numerical_cols),
    len(numerical_cols) + len(categorical_cols)
))

# One-hot encode categories after resampling.
after_sampling = ColumnTransformer([
    ("numerical", "passthrough", numerical_indices),
    ("categorical", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    ), categorical_indices)
])

smotenc_candidate = ImbPipeline([
    ("preprocessing", before_sampling),
    ("sampling", SMOTENC(
        categorical_features=categorical_indices,
        random_state=SEED,
        k_neighbors=5
    )),
    ("encoding", after_sampling),
    ("model", RandomForestClassifier(**rf_settings))
])

name = "Regularized + SMOTENC"
candidate_models[name] = smotenc_candidate

print(f"Evaluating: {name}", flush=True)

scores = cross_validate(
    smotenc_candidate,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    return_train_score=True,
    n_jobs=1,
    error_score="raise"
)

# Replace the row if this cell is rerun.
results = [row for row in results if row["Variety"] != name]

results.append({
    "Variety": name,
    "Parameters": str(smotenc_candidate.get_params()),
    "Train F1": scores["train_f1"].mean(),
    **{
        f"CV {metric}": scores[f"test_{metric}"].mean()
        for metric in scoring
    },
    "CV F1 std": scores["test_f1"].std(ddof=1)
})

comparison = pd.DataFrame(results).sort_values(
    "CV f1", ascending=False
)

display(comparison.drop(columns="Parameters").round(4))

Evaluating: Regularized + SMOTENC


,Variety,Train F1,CV f1,CV precision,CV recall,CV roc_auc,CV pr_auc,CV F1 std
4,Regularized + balanced,0.7335,0.6668,0.5720,0.8001,0.9235,0.7187,0.0100
9,Regularized + SMOTENC,0.7355,0.6657,0.5802,0.7811,0.9159,0.6620,0.0095
7,Regularized + undersampling,0.6730,0.6435,0.5183,0.8499,0.9207,0.7015,0.0109
5,Regularized + undersampling,0.6730,0.6435,0.5183,0.8499,0.9207,0.7015,0.0109
0,Baseline,1.0000,0.6137,0.7727,0.5092,0.9227,0.7369,0.0129
1,Balanced weights,1.0000,0.6066,0.7656,0.5026,0.9230,0.7311,0.0180
2,Balanced subsample,1.0000,0.6028,0.7701,0.4954,0.9243,0.7308,0.0194
3,Regularized trees,0.7236,0.5945,0.8286,0.4640,0.9254,0.7494,0.0191
6,"Regularized + balanced, no PageValues",0.5792,0.4239,0.3204,0.6265,0.7737,0.3823,0.0227
8,"Regularized + balanced, no PageValues",0.5792,0.4239,0.3204,0.6265,0.7737,0.3823,0.0227


In [10]:
from sklearn.model_selection import RandomizedSearchCV

# Remove repeated variety rows.
comparison = (
    pd.DataFrame(results)
    .drop_duplicates(subset="Variety", keep="last")
    .sort_values("CV f1", ascending=False)
    .reset_index(drop=True)
)
results = comparison.to_dict("records")

search_pipeline = clone(
    candidate_models["Regularized + balanced"]
)

# Parallelize CV fits; keep each forest single-threaded
# to avoid competing parallel jobs.
search_pipeline.set_params(model__n_jobs=1)

parameter_distributions = {
    "model__n_estimators": [100, 200, 300],
    "model__max_depth": [8, 12, 16, None],
    "model__min_samples_leaf": [2, 5, 10],
    "model__max_features": ["sqrt", 0.5, 1.0],
    "model__criterion": ["gini", "entropy"],
    "model__class_weight": ["balanced", "balanced_subsample"]
}

random_search = RandomizedSearchCV(
    estimator=search_pipeline,
    param_distributions=parameter_distributions,
    n_iter=24,
    scoring=scoring,
    refit="f1",
    cv=cv,
    random_state=SEED,
    n_jobs=-1,
    pre_dispatch="n_jobs",
    return_train_score=True,
    verbose=1,
    error_score="raise"
)

random_search.fit(X_train, y_train)

print("\nBest parameters:")
print(random_search.best_params_)
print(f"\nBest mean CV F1: {random_search.best_score_:.4f}")

search_results = pd.DataFrame(random_search.cv_results_)

columns = [
    "rank_test_f1",
    "mean_train_f1",
    "mean_test_f1",
    "std_test_f1",
    "mean_test_precision",
    "mean_test_recall",
    "mean_test_roc_auc",
    "mean_test_pr_auc",
    "params"
]

display(
    search_results
    .sort_values("rank_test_f1")[columns]
    .head(5)
    .reset_index(drop=True)
)

Fitting 5 folds for each of 24 candidates, totalling 120 fits

Best parameters:
{'model__n_estimators': 100, 'model__min_samples_leaf': 5, 'model__max_features': 0.5, 'model__max_depth': 12, 'model__criterion': 'gini', 'model__class_weight': 'balanced'}

Best mean CV F1: 0.6837


,rank_test_f1,mean_train_f1,mean_test_f1,std_test_f1,mean_test_precision,mean_test_recall,mean_test_roc_auc,mean_test_pr_auc,params
0,1,0.816849,0.683720,0.015180,0.617458,0.766707,0.931563,0.749431,"{'model__n_estimators': 100, 'model__min_sampl..."
1,2,0.815750,0.682004,0.011820,0.615556,0.765387,0.932669,0.754856,"{'model__n_estimators': 200, 'model__min_sampl..."
2,3,0.959266,0.681686,0.011820,0.683178,0.680855,0.929518,0.741207,"{'model__n_estimators': 200, 'model__min_sampl..."
3,4,0.861267,0.681496,0.014641,0.630468,0.742454,0.932776,0.754363,"{'model__n_estimators': 300, 'model__min_sampl..."
4,5,0.865686,0.681077,0.016442,0.630792,0.741149,0.928522,0.741768,"{'model__n_estimators': 100, 'model__min_sampl..."


In [ ]:
from sklearn.model_selection import GridSearchCV

refinement_pipeline = clone(random_search.best_estimator_)
refinement_pipeline.set_params(model__n_jobs=1)

refinement = GridSearchCV(
    estimator=refinement_pipeline,
    param_grid={
        "model__max_depth": [10, 12, 14],
        "model__min_samples_leaf": [3, 5, 7]
    },
    scoring=scoring,
    refit="f1",
    cv=cv,
    n_jobs=-1,
    pre_dispatch="n_jobs",
    return_train_score=True,
    verbose=1,
    error_score="raise"
)

refinement.fit(X_train, y_train)

print("\nRefined best parameters:")
print(refinement.best_params_)
print(f"\nRefined best CV F1: {refinement.best_score_:.4f}")

refinement_results = pd.DataFrame(refinement.cv_results_)

display(
    refinement_results.sort_values("rank_test_f1")[
        [
            "rank_test_f1",
            "mean_train_f1",
            "mean_test_f1",
            "std_test_f1",
            "mean_test_precision",
            "mean_test_recall",
            "mean_test_roc_auc",
            "mean_test_pr_auc",
            "params"
        ]
    ].reset_index(drop=True)
)

Fitting 5 folds for each of 9 candidates, totalling 45 fits
